In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True

In [3]:
# Parameters
location = "nigeria"
vehicle = "wheat"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if has_simulations:
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,2,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,2,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,2,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,2,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,9,0.0
26996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,9,0.0
26997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,9,0.0
26998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,9,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  488251.612663
                                  2                  381162.183392
                                  3                  383451.495271
                                  4                  374358.610912
                                  5                  271585.173914
intervention  maternal_disorders  1                  483796.045073
                                  2                  379147.417850
                                  3                  383451.495271
                                  4                  374358.610912
                                  5                  267614.332245
zero          maternal_disorders  1                  494660.339916
                                  2                  384568.688353
                                  3                  385417.930394
                                  4                  376563.007913
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,2,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,2,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,2,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,2,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
94495,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,9,0.0
94496,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,9,0.0
94497,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,9,0.0
94498,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,9,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  46768.230034
                                  2                  50529.265527
                                  3                  37324.045990
                                  4                  40581.273762
                                  5                  19654.755110
              maternal_disorders  1                  25228.269140
                                  2                  20675.751455
                                  3                  19650.009535
                                  4                  19641.509110
                                  5                  16017.475491
intervention  anemia              1                  46461.218053
                                  2                  50137.659362
                                  3                  36812.937367
                                  4                  40045.512672
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   46768.230034
                                  2                   50529.265527
                                  3                   37324.045990
                                  4                   40581.273762
                                  5                   19654.755110
              maternal_disorders  1                  513479.881803
                                  2                  401837.934847
                                  3                  403101.504806
                                  4                  394000.120022
                                  5                  287602.649405
intervention  anemia              1                   46461.218053
                                  2                   50137.659362
                                  3                   36812.937367
                                  4                   40045.512672
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # Combos with no iron pathway (e.g. ethiopia/salt, nigeria/salt) have no
    # Vivarium sims, so the corresponding DALYs are 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,334440.747771
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,337406.907859
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,296365.996516
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,269908.450785
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,193603.315485
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,292845.292479
1196,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,266909.197062
1197,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,266120.628846
1198,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,195712.180641


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.491293e+07
                      2                  1.549619e+07
                      3                  1.390872e+07
                      4                  1.230218e+07
                      5                  9.346356e+06
intervention  lbwsg   1                  1.491293e+07
                      2                  1.549325e+07
                      3                  1.390284e+07
                      4                  1.230218e+07
                      5                  9.346356e+06
zero          lbwsg   1                  1.491586e+07
                      2                  1.550206e+07
                      3                  1.390872e+07
                      4                  1.231099e+07
                      5                  9.349292e+06
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if has_anemia_model:
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,776.087681,zero
1,Female,0.0,0.019178,2,711.628699,zero
2,Female,0.0,0.019178,3,602.180380,zero
3,Female,0.0,0.019178,4,471.274914,zero
4,Female,0.0,0.019178,5,362.268907,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,272.358210,intervention
746,Male,95.0,125.000000,2,240.043729,intervention
747,Male,95.0,125.000000,3,242.054693,intervention
748,Male,95.0,125.000000,4,235.430039,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293973.956214
                      2                  291303.571496
                      3                  270955.169814
                      4                  262074.856772
                      5                  254222.010339
intervention  anemia  1                  292170.545928
                      2                  288783.146536
                      3                  267859.526636
                      4                  258623.999330
                      5                  250517.902167
zero          anemia  1                  299036.971776
                      2                  298379.582372
                      3                  279646.087074
                      4                  271763.026495
                      5                  264621.173225
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-47876.657233289676

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293835.928954
                      2                  269069.715819
                      3                  197667.182594
                      4                  156036.719905
                      5                  113104.494654
intervention  anemia  1                  292092.205538
                      2                  266926.969831
                      3                  195430.690955
                      4                  153999.357959
                      5                  111342.732542
zero          anemia  1                  298731.375574
                      2                  275085.405299
                      3                  203946.059931
                      4                  161756.547123
                      5                  118050.584326
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-27855.93032677693

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  979314.573276
                      2                  920878.151211
                      3                  758008.957731
                      4                  677280.676828
                      5                  590832.259510
intervention  anemia  1                  973324.886244
                      2                  913067.726051
                      3                  749246.151087
                      4                  668306.421824
                      5                  582037.449339
zero          anemia  1                  996130.424274
                      2                  942805.665122
                      3                  782610.251599
                      4                  702475.605069
                      5                  615523.402235
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if has_ntd_model:
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  5.013820e+05
                      2                  5.620520e+05
                      3                  6.657050e+05
                      4                  7.560143e+05
                      5                  1.058975e+06
baseline      ntd     1                  4.127409e+05
                      2                  4.220532e+05
                      3                  3.820743e+05
                      4                  3.386702e+05
                      5                  2.974603e+05
intervention  ntd     1                  3.893168e+05
                      2                  3.896594e+05
                      3                  3.365772e+05
                      4                  2.898707e+05
                      5                  2.465195e+05
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  1.026083e+06
                                  2                  9.714074e+05
                                  3                  7.953330e+05
                                  4                  7.178620e+05
                                  5                  6.104870e+05
              lbwsg               1                  1.491293e+07
                                  2                  1.549619e+07
                                  3                  1.390872e+07
                                  4                  1.230218e+07
                                  5                  9.346356e+06
              maternal_disorders  1                  5.134799e+05
                                  2                  4.018379e+05
                                  3                  4.031015e+05
                                  4                  3.940001e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)